# LiDAR quality exploration

This notebook acquires and inspects the measured PNOA-LiDAR evidence for one cadastral building. It stops before roof reconstruction: its outputs are the quality observations consumed by the reconstruction benchmark.


## Reproduce the LiDAR evidence

Change only `CADASTRAL_ROOT_ID` and run the notebook from the beginning. A clean **Run All** resolves Catastro and discovers, downloads, and crops the current third-coverage CNIG LiDAR asset automatically.


In [ ]:
import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

import laspy
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

from urbanstock3d.processors.lidar import points_in_polygon
from urbanstock3d.providers.pnoa_lidar import footprint_polygons_utm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN_INPUT_PIPELINE = True
# Change only this 14-character cadastral root to explore another building.
CADASTRAL_ROOT_ID = "4531917YJ2743B"
assert len(CADASTRAL_ROOT_ID) == 14, "Expected a 14-character cadastral root"
BUILDING_ID = f"ES.SDGC.BU.{CADASTRAL_ROOT_ID}"
BUILDING_PATH = PROJECT_ROOT / "outputs" / BUILDING_ID / "building.geojson"
CROP_PATH = PROJECT_ROOT / "outputs" / BUILDING_ID / "lidar_context_crop.laz"
QUALITY_PATH = PROJECT_ROOT / "outputs" / BUILDING_ID / "lidar_quality.json"
COVERAGE_PLOT_PATH = PROJECT_ROOT / "outputs" / BUILDING_ID / "lidar_coverage.png"

CLASS_STYLES = {
    1: ("Unclassified", "#7f7f7f"),
    2: ("Ground", "#8c564b"),
    3: ("Low vegetation", "#bcbd22"),
    4: ("Medium vegetation", "#2ca02c"),
    5: ("High vegetation", "#006400"),
    6: ("Building", "#d62728"),
    7: ("Noise", "#9467bd"),
    12: ("Legacy overlap", "#17becf"),
}
DEFAULT_CLASS_STYLE = ("Other", "#000000")
HEIGHT_COLORMAP = "viridis"

In [ ]:
# A clean Run All creates the cadastral geometry and LiDAR context before analysis.
if RUN_INPUT_PIPELINE:
    scripts_directory = PROJECT_ROOT / "scripts"
    urbanstock_executable = Path(sys.executable).with_name(
        "urbanstock.exe" if os.name == "nt" else "urbanstock"
    )
    commands = [
        [
            str(urbanstock_executable),
            "resolve",
            "--refcat",
            CADASTRAL_ROOT_ID,
            "--output-dir",
            str(PROJECT_ROOT / "outputs"),
        ],
        [
            sys.executable,
            str(scripts_directory / "process_lidar_crop.py"),
            str(BUILDING_PATH),
            "--buffer-m",
            "25",
            "--save-crop",
            str(CROP_PATH),
            "--output",
            str(PROJECT_ROOT / "outputs" / BUILDING_ID / "lidar_crop_audit.json"),
        ],
        [
            sys.executable,
            str(scripts_directory / "assess_lidar_quality.py"),
            str(CROP_PATH),
            str(BUILDING_PATH),
            "--output",
            str(QUALITY_PATH),
            "--plot",
            str(COVERAGE_PLOT_PATH),
        ],
    ]
    for command in commands:
        print(f"Running: {' '.join(command)}")
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Input pipeline skipped; existing artifacts will be used.")

## Review the production LiDAR quality report

These metrics are computed by the reusable quality assessor in `src`, not reimplemented in the notebook. Local PCA measures whether sampled neighbourhoods support planar roof geometry; it is evidence quality, not a reconstruction method.

In [ ]:
quality = json.loads(QUALITY_PATH.read_text(encoding="utf-8"))["quality"]
reported_metrics = {
    "Roof density (points/m2)": quality["density_roof_per_m2"],
    "Coverage at 0.5 m": quality["coverage_050m"],
    "Coverage at 1.0 m": quality["coverage_100m"],
    "Largest hole ratio": quality["largest_hole_ratio"],
    "Planar support ratio": quality["planar_support_ratio"],
    "Local residual median (m)": quality["local_residual_median_m"],
}
for name, value in reported_metrics.items():
    print(f"{name}: {value:.4f}")
print(f"Warnings: {quality['warnings'] or 'none'}")

coverage_image = plt.imread(COVERAGE_PLOT_PATH)
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(coverage_image)
ax.axis("off")
plt.show()

## 1. Load the reproducible context crop

In [ ]:
assert CROP_PATH.exists(), f"Generate the crop first: {CROP_PATH}"
assert BUILDING_PATH.exists(), f"Missing building geometry: {BUILDING_PATH}"
cloud = laspy.read(CROP_PATH)
x = np.asarray(cloud.x)
y = np.asarray(cloud.y)
z = np.asarray(cloud.z)
classification = np.asarray(cloud.classification, dtype=np.uint8)

print(f"Points: {len(cloud.points):,}")
print(f"CRS: {cloud.header.parse_crs()}")
print(f"Point format: {cloud.header.point_format.id}")
print(f"Dimensions: {list(cloud.point_format.dimension_names)}")
Counter(classification)

## 2. Inspect classifications in plan view

Class 12 is a legacy overlap classification in this tile, so it is visualized but excluded from geometric measurements.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
for class_id in np.unique(classification):
    mask = classification == class_id
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    ax.scatter(x[mask], y[mask], s=1, c=color, label=f"{label} ({mask.sum():,})")
ax.set(title="PNOA-LiDAR classifications", xlabel="Easting (m)", ylabel="Northing (m)")
ax.set_aspect("equal")
ax.legend(markerscale=5, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

## 3. Compare usable points and legacy overlap

In [ ]:
usable = classification != 12
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for class_id in np.unique(classification[usable]):
    mask = usable & (classification == class_id)
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    axes[0].scatter(x[mask], y[mask], c=color, s=2, label=label)
axes[0].set_title(f"Usable points ({usable.sum():,})")
axes[0].legend(markerscale=4)
overlap_label, overlap_color = CLASS_STYLES[12]
axes[1].scatter(x[~usable], y[~usable], c=overlap_color, s=2)
axes[1].set_title(f"{overlap_label} â€” class 12 ({(~usable).sum():,})")
for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
plt.show()

## 4. Normalize elevation against nearby classified ground

In [ ]:
ground = classification == 2
ground_z = float(np.median(z[ground]))
height = z - ground_z
print(f"Ground reference (median class 2): {ground_z:.2f} m")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
axes[0].hist(height[usable], bins=80, color="#7f7f7f")
axes[0].set(title="Ground-normalized height distribution", xlabel="Height (m)", ylabel="Points")
for class_id in np.unique(classification[usable]):
    mask = usable & (classification == class_id)
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    axes[1].scatter(x[mask], height[mask], c=color, s=2, label=label)
axes[1].set(title="East-height cross-section", xlabel="Easting (m)", ylabel="Height (m)")
axes[1].legend(markerscale=4)
plt.show()

## 5. Isolate classified roof points inside the cadastral footprint

In [ ]:
building = json.loads(BUILDING_PATH.read_text(encoding="utf-8"))
footprint_polygons = footprint_polygons_utm(building["geometry"])
in_footprint = np.zeros(x.shape, dtype=bool)
for rings in footprint_polygons:
    in_footprint |= points_in_polygon(x, y, rings)
roof = in_footprint & (classification == 6)
roof_height = height[roof]

print(f"Classified roof points in footprint: {roof.sum():,}")
print(f"Roof height p50: {np.percentile(roof_height, 50):.2f} m")
print(f"Roof height p95: {np.percentile(roof_height, 95):.2f} m")

fig, ax = plt.subplots(figsize=(9, 7))
plot = ax.scatter(x[roof], y[roof], c=roof_height, s=8, cmap=HEIGHT_COLORMAP)
ax.set(title="Roof points inside cadastral footprint", xlabel="Easting (m)", ylabel="Northing (m)")
ax.set_aspect("equal")
fig.colorbar(plot, ax=ax, label="Height above ground (m)")
plt.show()

## 6. Explore the point cloud interactively in 3D

The cloud is deterministically downsampled when necessary to keep interaction responsive. Each classification uses the same categorical color as the 2D views, while the vertical axis shows height above the local ground reference. Use the legend to show or hide individual classes.

In [ ]:
# Limit the displayed points to keep rotation and zoom responsive.
max_points = 30_000
indices = np.flatnonzero(usable)

if len(indices) > max_points:
    rng = np.random.default_rng(42)
    indices = rng.choice(indices, max_points, replace=False)

figure = go.Figure()
for class_id in np.unique(classification[indices]):
    class_indices = indices[classification[indices] == class_id]
    label, color = CLASS_STYLES.get(int(class_id), DEFAULT_CLASS_STYLE)
    figure.add_trace(
        go.Scatter3d(
            x=x[class_indices],
            y=y[class_indices],
            z=height[class_indices],
            mode="markers",
            name=f"{label} â€” class {class_id}",
            marker={"size": 1.5, "color": color, "opacity": 0.8},
            customdata=np.full(len(class_indices), class_id),
            hovertemplate=(
                "Easting: %{x:.2f} m<br>"
                "Northing: %{y:.2f} m<br>"
                "Height: %{z:.2f} m<br>"
                "Class: %{customdata}<extra>%{fullData.name}</extra>"
            ),
        )
    )

figure.update_layout(
    title="Ground-normalized LiDAR point cloud",
    scene={
        "xaxis_title": "Easting (m)",
        "yaxis_title": "Northing (m)",
        "zaxis_title": "Height above ground (m)",
        "aspectmode": "data",
    },
    legend={"title": "PNOA-LiDAR class"},
    height=750,
)

figure.show()

## Observations

Record conclusions about classification quality, usable density, ground normalization, roof support, vegetation, outliers, and spatial gaps. Reconstruction decisions belong in `02_reconstruction_benchmark.ipynb`.
